In [1]:
import torch
from util import TokenizerUtil

tokenizer = TokenizerUtil()

input_ids, _ = tokenizer.encode('how are you', max_length=6)

input_ids, attention_mask = tokenizer.pad_to_left(input_ids)

input_ids, attention_mask, tokenizer.decode(input_ids)

(tensor([   1,    1,    0, 9178,   32,   47]),
 tensor([0, 0, 1, 1, 1, 1]),
 '<pad><pad><s>how are you')

加载数据集， 只要问题， 不用回答

In [2]:
from datasets import load_dataset
from transformers import default_data_collator, get_scheduler

dataset = load_dataset(
    'json',
    data_files = './dataset/train.json',
    split="train"
)

dataset = dataset.select(range(45000, len(dataset)))

def f(data):

    # 这里数据集构造的时候， prompt部分最长是256
    input_ids, _ = tokenizer.encode(data['prompt'], max_length=256)
    input_ids, attention_mask = tokenizer.pad_to_left(input_ids)

    return {'input_ids': input_ids, 'attention_mask':attention_mask}

dataset = dataset.map(f, remove_columns = dataset.column_names)

loader = torch.utils.data.DataLoader(dataset, 
                                    collate_fn = default_data_collator,
                                    batch_size=4,
                                    shuffle=True,
                                    drop_last=True)

len(loader), next(iter(loader))

(7144,
 {'input_ids': tensor([[    1,     1,     1,  ...,   116,  6267,    35],
          [    1,     1,     1,  ..., 15496,  6267,    35],
          [    1,     1,     1,  ...,   116,  6267,    35],
          [    1,     1,     1,  ...,   116,  6267,    35]]),
  'attention_mask': tensor([[0, 0, 0,  ..., 1, 1, 1],
          [0, 0, 0,  ..., 1, 1, 1],
          [0, 0, 0,  ..., 1, 1, 1],
          [0, 0, 0,  ..., 1, 1, 1]])})

# 1. 加载演员模型

In [3]:
from transformers import AutoModelForCausalLM
import lora

model_actor = AutoModelForCausalLM.from_pretrained('./model/actor', dropout=0.0)
lora.insert(model_actor)

def getparams():
    params = []
    params_lora = []
    for name, p in model_actor.named_parameters():
        if not p.requires_grad:
            continue
        if 'lora_A' in name or 'lora_B' in name:
            params_lora.append(p)
            continue

        params.append(p)

    return [
        {'params':params,
        'weight_decay':0.0},
        
        {'params':params_lora,
        'weight_decay':0.0,
        'lr':5e-4}
    ]

Loading weights:   0%|          | 0/388 [00:00<?, ?it/s]

In [4]:
optimizer_actor = torch.optim.Adam(getparams(), lr=1e-5, betas=(0.9, 0.95))

# 学习率调度器配置
scheduler_actor = get_scheduler(name = 'cosine',
                               optimizer = optimizer_actor,
                               num_warmup_steps = 100,
                               num_training_steps=800)

model_actor.train()

lora.count_params(model_actor)

{'count_require': 2.21044736, 'count_all': 14.29004288, 'ratio': 0.15468444556549854}


# 2. 加载Critic模型

In [5]:
class CriticModel(torch.nn.Module):
    def __init__(self):
        super().__init__()

        from transformers import AutoModel
        self.rwtransformer = AutoModel.from_pretrained('./opt-350m', dropout=0.0)

        self.v_head = torch.nn.Linear(512, 1, bias=False).to(torch.bfloat16)

    def get_value(self, input_ids, attention_mask):
        value = self.rwtransformer(
            input_ids = input_ids,
            attention_mask = attention_mask
        ).last_hidden_state
        return self.v_head(value).squeeze(-1)

    def get_reward(self, input_ids, attention_mask):
        value = self.get_value(input_ids, attention_mask)
        reward = []

        for i, v in zip(input_ids, value):
            end = input_ids.shape[1] - 1
            if tokenizer.eos_token_id in i :
                end = i.tolist().index(tokenizer.eos_token_id)
            reward.append(v[end])
        reward = torch.stack(reward, dim=0)
        return reward

In [6]:
model_critic = torch.load("model/critic", weights_only=False)

optimizer_critic = torch.optim.Adam(model_critic.parameters(),
                                   lr=5e-6,
                                   betas=(0.9, 0.95))

scheduler_critic = get_scheduler(name='cosine',
                                optimizer=optimizer_critic,
                                num_warmup_steps=100,
                                num_training_steps=800)

model_critic.train()

lora.count_params(model_critic)

{'count_require': 3.31196928, 'count_all': 3.31196928, 'ratio': 1.0}


# 3. 加载ref 和 reward 

In [7]:
from accelerate import Accelerator

model_ref = AutoModelForCausalLM.from_pretrained('model/actor', weights_only=False)
model_reward = torch.load('model/critic',weights_only=False)

model_ref.eval()
model_reward.eval()

accelerator = Accelerator(gradient_accumulation_steps = 1,
                         mixed_precision='bf16')

(loader,
 model_actor, optimizer_actor, scheduler_actor, 
 model_critic, optimizer_critic, scheduler_critic) = accelerator.prepare(loader, 
                                                                         model_actor, optimizer_actor, scheduler_actor, 
                                                                         model_critic, optimizer_critic, scheduler_critic)

Loading weights:   0%|          | 0/388 [00:00<?, ?it/s]

In [8]:
def get_generate(input_ids, attention_mask):

    # actor_model 用来生成预测结果 [batch, sequence_length]
    generate = model_actor.generate(input_ids,
                                   attention_mask = attention_mask,
                                   max_length = 512,
                                   pad_token_id = tokenizer.pad_token_id,
                                   eos_token_id = tokenizer.eos_token_id)

    # 这里前面256是prompt， 然后后面从256开始是机器生成的回答，然后找到不是pad的长度
    lens = (generate[:, 256:] != tokenizer.pad_token_id).sum(1)

    # 这里选取机器生成回答是大于1的有效回答
    return generate[lens > 1]

data = next(iter(loader))

get_generate(**data).shape


torch.Size([4, 296])

In [9]:
def get_prob(prob, index):
    # 这里首先是对最后一个维度进行softmax
    prob = prob.log_softmax(dim=2)

    # 这里index是扩张一个维度， 因为gather要求index和prob的索引一致，收集到各个index对应位置的索引后，再进行缩小回去
    prob = prob.gather(dim=2, index=index.unsqueeze(2))
    return prob.squeeze(2)

# 获取PPO训练数据 

In [10]:
last_generate = None

def get_batch(input_ids, attention_mask):
    """
    用于制造PPO的训练数据
    input_ids : [batchsize, sequence_length] = [4, 256]
    attentionamsk : [batchsize, sequence_length] = [4, 256]

    return :
    1. 
    """
    global last_generate

    # 这一步直接调用actor生成回答， 即[batch_size, response_length]
    generate = get_generate(input_ids, attention_mask)

    #制作缓存,防止所有回答为空的情况
    if len(generate):
        last_generate = generate
    else:
        generate = last_generate

    # prompt+response部分的mask 即[batch_size, response_length]
    generate_mask = (generate != tokenizer.pad_token_id).long()

    # 1. 调用actor， 查看每个词语生成的概率， [batch_size, response_length-1]
    prob_old = model_actor(input_ids = generate,
                          attention_mask = generate_mask).logits
    prob_old = get_prob(prob_old[:, :-1], generate[:, 1:]) # 这句话是找到生成这些词语， 对应的概率是多少

    # 2. 调用critic模型对每个词语的评分：[batch_size, response_length-1]
    value_old = model_critic.get_value(generate, generate_mask)[:, :-1]

    # 3. 使用reference模型获取， 生成这些词语的对应概率[batch_size, response_length-1]
    prob_ref = model_ref(
        input_ids = generate.to('cpu'),
        attention_mask = generate_mask.to('cpu')
    ).logits.to('cuda')
    prob_ref = get_prob(prob_ref[:, :-1], generate[:, 1:])

    # 4. 使用reward对每个样本进行一个奖励评估, [batch_size] 
    reward = model_reward.get_reward(generate.to('cpu'),
                                    generate_mask.to('cpu')).to('cuda')

    return generate, generate_mask, prob_old, prob_ref, value_old, reward

In [11]:
generate, generate_mask, prob_old, prob_ref, value_old, reward = get_batch(**data)

generate.shape, generate_mask.shape, prob_old.shape, prob_ref.shape, value_old.shape, reward.shape

(torch.Size([4, 296]),
 torch.Size([4, 296]),
 torch.Size([4, 295]),
 torch.Size([4, 295]),
 torch.Size([4, 295]),
 torch.Size([4]))

In [12]:
def get_reward_kl(end, prob_old, prob_ref, reward):
    """
    这个函数：计算两个概率向量的距离，然后将每句话的reward直接加到最后一个词语当中
    end ：【batchsize】 主要是记录各个句子的最后一个有效token的位置
    prob_old -> [4, gen_lens-1]
    prob_ref -> [4, gen_lens-1]
    reward -> [4]

    returns : 
    1. [4, gen_lens-1]
    """

    # 这里KL散度作为一个惩罚项， 然后在强化学习当中是为了最大化目标函数， 因此这里KL散度最后需要取反，作为惩罚
    reward_kl = -0.1 * (prob_old - prob_ref)

    # 将reward 加到kl三都的最后一个词语上
    for index, end_token_position in enumerate(end):
        if end_token_position >= reward_kl.shape[1]:
            end_token_position = -1 # 超出索引就直接填最后一个词语

        reward_kl[index, end_token_position] += reward[index].clamp(-5, 5)

    return reward_kl

In [13]:
end = generate_mask[:, 256:].sum(1) + 255
end = end.tolist()

reward_kl = get_reward_kl(end, prob_old, prob_ref, reward)

reward_kl.shape

torch.Size([4, 295])

value_old：对应教程中的 $V_t$，即 Critic 模型预测的 $t$ 时刻的总收益预估。

reward_kl：对应教程中的 $r_t$，即 $t$ 时刻的即时收益（这其中已经包含了 Reward 模型的打分和 Reference 模型的 KL 散度约束）。

value_next ： 对应教程中的 $V_{t+1}$，即 Critic 模型预测的 $t+1$ 时刻的总收益预估。

$$单步优势\delta_t = r_t + \gamma V_{t+1} - V_t$$

1. Critic模型生成生成出来的value[:, i] 代表采用动作第i这个字符（即动作）以及后续的所有字符加起来会对结果产生多少好处
2. 因此value[:, i+1] - value[:, i]代表Critic判断采用i位置这个字符会带来的好处
3. 再加上kl散度的惩罚项， 就可以得到单步优势

当前所有优势 = 当前单步优势 + 预期未来优势（打折扣，因为未来不确定）
$$A_t = \delta_t + \gamma \lambda A_{t+1}$$



In [14]:
def get_advantage(value_old, reward_kl):
    """
    这里优势计算函数
    #value_old -> [4, gen_lens-1]
    #reward_kl -> [4, gen_lens-1]

    value_old 是 critic预计从当前这步开始到后面所带来的所有收益
    value_next 代表critic预计从i+1 一直到最后带来的收益
    reward_kl 是 actor 和 reference 共同参考下生成出来kl散度惩罚项，用于限制actor生成离谱答案
    
    """
    # 这里记录的是每个action的预期优势
    advantages = []
    for i in reversed(range(255, value_old.shape[1])):
        value_next = 0.0
        if i == value_old.shape[1]-1:
            value_next = 0.0 # 如果i是最后一个词语， 那么后面没有动作，那么也就不会带来收益
        else :
            value_next = value_old[:, i+1] # value_next代表后面带来的收益

        single_action_advantage = reward_kl[:, i] + value_next - value_old[:, i] # 单步所带来的收益

        if(len(advantages)):
            single_action_advantage += advantages[-1]*0.95 # 这里advantages还是倒序，所以这里相当于当前优势 + 未来优势*0.95
            # single_action_advantage [4]

        advantages.append(single_action_advantage)

    advantages = torch.stack(advantages[::-1], dim=1) # 变成[4, gen_lens-256]

    return advantages

In [15]:
advantages = get_advantage(value_old, reward_kl)

advantages.shape

torch.Size([4, 40])

In [16]:
def get_loss_actor(prob_new, prob_old, delta, generate_mask):
    """"
    这里delta 是 GAE
    generate_mask
    """
    prob_new = prob_new[:, 255:]
    prob_old = prob_old[:, 255:]
    generate_mask = generate_mask[:, 256:] 

    # prob_new / prob_old, 这里比例直接除就好了, 由于是log, 因此直接相减,  
    ratio = ((prob_new - prob_old) * generate_mask).exp()

    # 这里拉伸到比例就是
    loss1 = delta * ratio
    loss2 = delta * ratio.clamp(0.8, 1.2)
    loss = torch.min(loss1, loss2) * generate_mask

    # 这里会把整个batch到loss全部会汇聚到一起
    loss = loss.sum() / generate_mask.sum() / 8
    return -loss

In [17]:
def get_loss_critic(value_new, value_old, delta, generate_mask):
    value_new = value_new[:, 255:]
    value_old = value_old[:, 255:]
    generate_mask = generate_mask[:, 256:]

    #value_new -> [4, gen_lens-256]
    #value_old -> [4, gen_lens-256]
    #delta -> [4, gen_lens-256]
    #generate_mask -> [4, gen_lens-256]

    #delta是估计出来的去基线Q值,加上value_old后还原为Q值
    #value_new和Q值求mse loss即可,因为value都是对Q函数的估计
    #裁剪,防止自举
    #[4, gen_lens-256]
    loss1 = (value_new - delta - value_old)**2
    value_new = value_new.clamp(value_old - 0.2, value_old + 0.2)
    loss2 = (value_new - delta - value_old)**2

    #求平均
    loss = torch.max(loss1, loss2) * generate_mask
    loss = loss.sum() / 2 / generate_mask.sum() / 8

    return loss


loss_critic = get_loss_critic(value_old, value_old, advantages, generate_mask)

loss_critic

tensor(0.3224, device='cuda:0', grad_fn=<DivBackward0>)

In [18]:
def train(generate, generate_mask, prob_old, prob_ref, value_old, reward,
          do_step):
    #generate -> [4, gen_lens]
    #generate_mask -> [4, gen_lens]
    #prob_old -> [4, gen_lens-1]
    #prob_ref -> [4, gen_lens-1]
    #value_old -> [4, gen_lens-1]
    #reward -> [4]
    #do_step -> bool

    #求出每句话结束的索引
    #[4]
    end = generate_mask[:, 256:].sum(1) + 255
    end = end.tolist()

    #结束以后的value归零
    for i, e in enumerate(end):
        value_old[i, e + 1:] = 0

    with torch.no_grad():
        #计算新旧概率的kl散度,再把reward加在最后一个字上
        #[4, gen_lens-1]
        reward_kl = get_reward_kl(end, prob_old, prob_ref, reward)

        #估计去基线的Q值
        #[4, gen_lens-256]
        delta = get_advantage(value_old, reward_kl)

    #重新计算回答被生成的概率
    #[4, gen_lens-1]
    prob_new = model_actor(input_ids=generate,
                           attention_mask=generate_mask).logits
    prob_new = get_prob(prob_new[:, :-1], generate[:, 1:])

    #更新actor
    loss_actor = get_loss_actor(prob_new, prob_old, delta, generate_mask)
    accelerator.backward(loss_actor)
    if do_step:
        accelerator.clip_grad_norm_(
            [i for i in model_actor.parameters() if i.requires_grad], 1.0)
        optimizer_actor.step()
        scheduler_actor.step()
        optimizer_actor.zero_grad()

    #重新计算每个词的value
    #[4, gen_lens-1]
    value_new = model_critic.get_value(input_ids=generate,
                                       attention_mask=generate_mask)[:, :-1]

    #更新critic
    loss_critic = get_loss_critic(value_new, value_old, delta, generate_mask)
    accelerator.backward(loss_critic)
    if do_step:
        accelerator.clip_grad_norm_(model_critic.parameters(), 1.0)
        optimizer_critic.step()
        scheduler_critic.step()
        optimizer_critic.zero_grad()

    return loss_actor.item(), loss_critic.item()


train(generate, generate_mask, prob_old, prob_ref, value_old, reward, True)

(0.06879522651433945, 0.3167209029197693)

In [19]:
for i, data in enumerate(loader):
    #生成数据
    (generate, generate_mask, prob_old, prob_ref, value_old,
     reward) = get_batch(**data)

    do_step = (i + 1) % 8 == 0

    #训练
    loss_actor, loss_critic = train(generate, generate_mask, prob_old,
                                    prob_ref, value_old, reward, do_step)

    if do_step:
        lr_actor = optimizer_actor.param_groups[0]['lr']
        lr_critic = optimizer_critic.param_groups[0]['lr']
        print(i, len(loader), loss_actor, loss_critic, lr_actor, lr_critic)
        print(tokenizer.decode(generate[0, 256:]))
        print(reward[0].item())

    if i == 2500:
        break

lora.merge(model_actor)
model_actor.save_pretrained('model/rlhf')

7 7144 0.08916010707616806 0.34631913900375366 2.0000000000000002e-07 1.0000000000000001e-07
select OPPONENT from TABLE_NAME_70 where SCORE = "1-3"</s>
3.859375
15 7144 0.042368609458208084 0.46839016675949097 3.0000000000000004e-07 1.5000000000000002e-07
SELECT drawn FROM table_name_86 WHERE played < 16 AND tries < 1 AND %WON < 56.15%</s>
-5.65625
23 7144 0.07386516779661179 0.2470327913761139 4.0000000000000003e-07 2.0000000000000002e-07
select count(cover_model) from TABLE_NAME_8 where DATE = "8-03"</s>
4.375
31 7144 -0.014736663550138474 0.49114012718200684 5.000000000000001e-07 2.5000000000000004e-07
SELECT floor_area FROM table_name_41 WHERE street_address = "921 sw sixth avenue"</s>
-5.75
39 7144 0.01505710743367672 0.4589022696018219 6.000000000000001e-07 3.0000000000000004e-07
select THIRD from TABLE_NAME_47 where SKIP = "BORA VOJTUSOVA"</s>
5.28125
47 7144 0.11119852215051651 0.273831307888031 7.000000000000001e-07 3.5000000000000004e-07
select FIRST_ELECTED from TABLE_134145

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 23.57 GiB of which 13.50 MiB is free. Including non-PyTorch memory, this process has 15.86 GiB memory in use. Process 1460 has 7.68 GiB memory in use. Of the allocated memory 15.43 GiB is allocated by PyTorch, and 118.79 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)